# ML-10 — Content Action Playbook

This notebook turns the validated Week-5 Random Forest output into a practical, human-reviewed content action queue.

The queue is for decision support, not automatic publishing. A high score means the model found the page worth reviewing for decline risk; it does not prove why the page is declining or guarantee that an action will improve performance.

Executed through the ML-10 workflow for reproducible submission.


## 1. Intended use

The playbook gives a content team a short, ranked list of pages to review first.

Each row contains:
- decline-risk score
- content archetype
- recommended action
- reason code based on observable signals
- human-review requirement
- simple cost/value priority

The final action must be confirmed by a human. The model should not publish, delete, redirect, or rewrite content automatically.


In [1]:
from pathlib import Path
import os, json
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

DATA_PATH = Path(os.environ.get("FLYRANK_DATA_PATH", "data/raw/content_refresh_anonymized.csv"))
if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    DATA_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
    df = pd.read_csv(DATA_URL)

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates("content_id").reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Rows used: {len(df):,}")
print(f"Clients: {df.client_id.nunique():,}")
print(f"Decline base rate: {df.is_declining_label.mean():.2%}")


Rows used: 30,000
Clients: 32
Decline base rate: 54.21%


## 2. Rebuild the validated client-grouped model

ML-09 showed that the honest client-grouped split is the primary validation design because test clients are completely separate from training clients.

The same split and Random Forest settings are reused here so the action queue comes from the validated model rather than a different experiment.


In [2]:
numeric_features = [
    "search_volume","cpc","word_count","char_count","impressions_90d",
    "clicks_90d","sessions_90d","ai_sessions_90d",
    "days_with_impressions_90d","days_with_sessions_90d",
    "content_age_days","days_since_last_update","ctr","avg_position",
    "engagement_rate","scroll_rate","ai_traffic_pct"
]
categorical_features = [
    "competition","content_type","main_intent","competition_level",
    "age_tier","freshness_tier","word_count_tier","impression_tier","position_tier"
]
features = [c for c in numeric_features + categorical_features if c in df.columns]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, y=df["is_declining_label"], groups=df["client_id"]))
train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

def build_model():
    nums = [c for c in numeric_features if c in df.columns]
    cats = [c for c in categorical_features if c in df.columns]
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), nums),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cats)
    ])
    return Pipeline([
        ("prep", prep),
        ("rf", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=5,
            class_weight="balanced_subsample", random_state=42, n_jobs=-1
        ))
    ])

model = build_model()
model.fit(train[features], train["is_declining_label"])
test["decline_risk_score"] = model.predict_proba(test[features])[:, 1]

assert set(train.client_id).isdisjoint(set(test.client_id))
print(f"Train rows: {len(train):,}")
print(f"Test rows: {len(test):,}")
print(f"Train clients: {train.client_id.nunique()}")
print(f"Test clients: {test.client_id.nunique()}")
print("Client overlap:", len(set(train.client_id) & set(test.client_id)))


Train rows: 23,837
Test rows: 6,163
Train clients: 25
Test clients: 7
Client overlap: 0


## 3. Reason codes and content archetypes

The reason codes are descriptive. They identify visible signals in the data; they do not claim that a signal caused the decline.

Percentile ranks are calculated inside the held-out test set so the rules stay tied to the current evaluation population.


In [3]:
def pct_rank(s):
    return s.rank(method="average", pct=True).fillna(0.5)

test["impressions_pct"] = pct_rank(np.log1p(test["impressions_90d"]))
test["freshness_pct"] = pct_rank(test["days_since_last_update"])
test["position_pct"] = pct_rank(test["avg_position"].clip(lower=1, upper=100))
test["age_pct"] = pct_rank(test["content_age_days"])
test["depth_pct"] = pct_rank(test["word_count"])
test["missing_key_signal"] = test[["word_count","avg_position","impressions_90d"]].isna().any(axis=1)

def reason_codes(r):
    codes = []
    if r["freshness_pct"] >= 0.75:
        codes.append("STALE")
    if r["position_pct"] >= 0.75 and r["impressions_pct"] >= 0.50:
        codes.append("RANKING_OPPORTUNITY")
    if pd.notna(r["word_count"]) and r["depth_pct"] <= 0.25:
        codes.append("DEPTH_GAP")
    if r["age_pct"] >= 0.75:
        codes.append("MATURE_CONTENT")
    if r["missing_key_signal"]:
        codes.append("MISSING_SIGNAL")
    return "|".join(codes) if codes else "MODEL_RISK_ONLY"

test["reason_code"] = test.apply(reason_codes, axis=1)

def choose_archetype(r):
    if "MISSING_SIGNAL" in r["reason_code"]:
        return "Needs data review"
    if "STALE" in r["reason_code"] and "RANKING_OPPORTUNITY" in r["reason_code"]:
        return "Stale high-visibility page"
    if "RANKING_OPPORTUNITY" in r["reason_code"]:
        return "Ranking opportunity"
    if "DEPTH_GAP" in r["reason_code"]:
        return "Depth improvement candidate"
    if "STALE" in r["reason_code"]:
        return "Refresh candidate"
    return "General decline-risk review"

test["archetype"] = test.apply(choose_archetype, axis=1)

action_map = {
    "Stale high-visibility page": "Refresh and revalidate",
    "Ranking opportunity": "Review search intent and on-page relevance",
    "Depth improvement candidate": "Review content depth and coverage",
    "Refresh candidate": "Review freshness and update need",
    "Needs data review": "Resolve missing or inconsistent signals first",
    "General decline-risk review": "Human content review"
}
test["recommended_action"] = test["archetype"].map(action_map)

test["priority_score"] = (
    0.60 * test["decline_risk_score"] +
    0.20 * test["impressions_pct"] +
    0.20 * (1 - test["position_pct"])
).clip(0, 1)

test["human_review"] = "Required"
test["automation_allowed"] = "No"


## 4. Ranked action queue

The queue is sorted by priority, but the first row is not automatically the best business decision.

Before acting, a reviewer should confirm the page intent, business importance, current search-result context, recent changes, and whether the suggested action is actually feasible.


In [4]:
queue_cols = [
    "content_id","client_id","decline_risk_score","priority_score",
    "archetype","recommended_action","reason_code",
    "content_age_days","days_since_last_update","impressions_90d","impressions_pct",
    "avg_position","word_count","human_review","automation_allowed"
]
queue = test.sort_values(
    ["priority_score","decline_risk_score"], ascending=False
)[queue_cols].copy()

queue["decline_risk_score"] = queue["decline_risk_score"].round(3)
queue["priority_score"] = queue["priority_score"].round(3)

print("Top 10 action queue:")
display(queue.head(10))
print(f"Queue rows: {len(queue):,}")


Top 10 action queue:


,content_id,client_id,decline_risk_score,priority_score,archetype,recommended_action,reason_code,content_age_days,days_since_last_update,impressions_90d,impressions_pct,avg_position,word_count,human_review,automation_allowed
13215,content_98aa0aecb1d9,client_8527a891e2,0.898,0.877,Depth improvement candidate,Review content depth and coverage,STALE|DEPTH_GAP,275,104,5190,0.846828,5.1,1422.0,Required,No
3211,content_f55fd2d8ed04,client_4e07408562,0.858,0.858,Depth improvement candidate,Review content depth and coverage,STALE|DEPTH_GAP,230,104,2237,0.736330,1.3,1604.0,Required,No
4076,content_66458ac1b739,client_8527a891e2,0.811,0.852,Depth improvement candidate,Review content depth and coverage,STALE|DEPTH_GAP,223,102,6822,0.875710,2.9,1506.0,Required,No
18063,content_b08562686d22,client_f369cb89fc,0.839,0.852,Refresh candidate,Review freshness and update need,STALE,106,106,2846,0.771215,2.0,2900.0,Required,No
18631,content_cb2d43abb7f6,client_f369cb89fc,0.839,0.852,General decline-risk review,Human content review,MODEL_RISK_ONLY,131,20,2931,0.777787,2.4,3093.0,Required,No
14343,content_9ac61c04930e,client_8527a891e2,0.899,0.841,Depth improvement candidate,Review content depth and coverage,STALE|DEPTH_GAP,275,104,1828,0.704040,5.6,1504.0,Required,No
4050,content_500bd3907331,client_4e07408562,0.857,0.841,Depth improvement candidate,Review content depth and coverage,STALE|DEPTH_GAP,230,104,4037,0.818108,5.5,1294.0,Required,No
4691,content_afa114dc512c,client_f369cb89fc,0.770,0.839,General decline-risk review,Human content review,MODEL_RISK_ONLY,97,20,10277,0.916599,2.2,2873.0,Required,No
2509,content_71efc27b02b3,client_4e07408562,0.864,0.839,Depth improvement candidate,Review content depth and coverage,STALE|DEPTH_GAP,230,104,1521,0.677835,3.7,1510.0,Required,No
13572,content_ccf887ee3581,client_4e07408562,0.857,0.833,Depth improvement candidate,Review content depth and coverage,STALE|DEPTH_GAP,326,104,2993,0.781438,5.5,1377.0,Required,No


Queue rows: 6,163


## 5. Cost/value thinking

The dataset does not contain real business costs, so the playbook does not pretend to calculate financial ROI.

Instead, it uses a simple prioritisation note:
- high decline risk + meaningful visibility → review sooner
- missing key signals → data review first
- other cases → normal human review

These are prioritisation aids, not measured financial returns.


In [5]:
def cost_value_label(r):
    if r["archetype"] == "Needs data review":
        return "LOW COST / GATING REVIEW"
    if r["impressions_pct"] >= 0.75 and r["decline_risk_score"] >= 0.75:
        return "HIGH POTENTIAL / REVIEW FIRST"
    if r["decline_risk_score"] >= 0.75:
        return "MEDIUM-HIGH / REVIEW"
    return "NORMAL REVIEW"

queue["cost_value_note"] = queue.apply(cost_value_label, axis=1)
display(queue.head(10)[[
    "content_id","decline_risk_score","archetype",
    "recommended_action","cost_value_note"
]])


,content_id,decline_risk_score,archetype,recommended_action,cost_value_note
13215,content_98aa0aecb1d9,0.898,Depth improvement candidate,Review content depth and coverage,HIGH POTENTIAL / REVIEW FIRST
3211,content_f55fd2d8ed04,0.858,Depth improvement candidate,Review content depth and coverage,MEDIUM-HIGH / REVIEW
4076,content_66458ac1b739,0.811,Depth improvement candidate,Review content depth and coverage,HIGH POTENTIAL / REVIEW FIRST
18063,content_b08562686d22,0.839,Refresh candidate,Review freshness and update need,HIGH POTENTIAL / REVIEW FIRST
18631,content_cb2d43abb7f6,0.839,General decline-risk review,Human content review,HIGH POTENTIAL / REVIEW FIRST
14343,content_9ac61c04930e,0.899,Depth improvement candidate,Review content depth and coverage,MEDIUM-HIGH / REVIEW
4050,content_500bd3907331,0.857,Depth improvement candidate,Review content depth and coverage,HIGH POTENTIAL / REVIEW FIRST
4691,content_afa114dc512c,0.770,General decline-risk review,Human content review,HIGH POTENTIAL / REVIEW FIRST
2509,content_71efc27b02b3,0.864,Depth improvement candidate,Review content depth and coverage,MEDIUM-HIGH / REVIEW
13572,content_ccf887ee3581,0.857,Depth improvement candidate,Review content depth and coverage,HIGH POTENTIAL / REVIEW FIRST


## 6. Human review and no-go rules

Every recommended action requires human review.

### Review before action
- Confirm the page's search intent and business purpose.
- Check recent edits and known traffic changes.
- Check the current search-result context before changing the page.
- Confirm that the recommendation matches the actual content problem.
- Record what was changed so the outcome can be checked later.

### No-go cases
Do not automate:
- publishing or rewriting content
- deleting pages
- redirects or canonical changes
- changes to important legal, medical, financial, or brand-sensitive claims
- actions based mainly on missing or unreliable data
- claims that the model has identified the cause of decline

The model is a prioritisation tool, not an autonomous content operator.


## 7. Monitoring and retraining triggers

Review the workflow when:
- the client mix changes materially
- feature distributions shift noticeably
- the observed decline rate changes materially
- top-queue precision drops after outcomes become available
- new content types or business areas are introduced
- a later client-grouped validation run shows performance deterioration

Retraining should be considered after enough new labelled outcomes are available for another honest client-grouped evaluation. A calendar date alone is not proof that retraining is needed.


In [6]:
out_dir = Path("work/outputs")
metrics_path = Path("work/ml10_metrics.json")
out_dir.mkdir(parents=True, exist_ok=True)

queue_path = out_dir / "ml10_ranked_action_queue.csv"
queue.to_csv(queue_path, index=False)

metrics = {
    "rows_used": int(len(df)),
    "clients": int(df.client_id.nunique()),
    "train_clients": int(train.client_id.nunique()),
    "test_clients": int(test.client_id.nunique()),
    "client_overlap": int(len(set(train.client_id) & set(test.client_id))),
    "decline_base_rate": round(float(df.is_declining_label.mean()), 4),
    "queue_rows": int(len(queue)),
    "high_risk_top_10": int((queue.head(10).decline_risk_score >= 0.75).sum()),
    "automation_allowed": False,
    "human_review_required": True,
    "model": "RandomForestClassifier",
    "n_estimators": 300,
    "min_samples_leaf": 5,
    "random_state": 42
}
metrics_path.write_text(json.dumps(metrics, indent=2))

print(f"Exported: {queue_path}")
print(f"Exported: {metrics_path}")
print(json.dumps(metrics, indent=2))


Exported: work/outputs/ml10_ranked_action_queue.csv
Exported: work/ml10_metrics.json
{
  "rows_used": 30000,
  "clients": 32,
  "train_clients": 25,
  "test_clients": 7,
  "client_overlap": 0,
  "decline_base_rate": 0.5421,
  "queue_rows": 6163,
  "high_risk_top_10": 10,
  "automation_allowed": false,
  "human_review_required": true,
  "model": "RandomForestClassifier",
  "n_estimators": 300,
  "min_samples_leaf": 5,
  "random_state": 42
}


## 8. Final self-check

The assignment is complete when the notebook has:
1. ranked actions with reason codes
2. mapped content archetypes to actions
3. explained intended use and limits
4. required human review and listed no-go cases
5. included monitoring and retrain triggers
6. included practical cost/value thinking
7. exported the ranked queue for the paper
8. kept the workflow non-production and non-automated


In [7]:
checks = {
    "ranked_actions": len(queue) > 0,
    "reason_codes": queue["reason_code"].notna().all(),
    "archetype_action_mapping": queue["recommended_action"].notna().all(),
    "human_review_required": (queue["human_review"] == "Required").all(),
    "automation_disabled": (queue["automation_allowed"] == "No").all(),
    "zero_client_overlap": len(set(train.client_id) & set(test.client_id)) == 0,
    "queue_exported": queue_path.exists(),
    "metrics_exported": metrics_path.exists(),
}
for k, v in checks.items():
    print(f"{k}: {'PASS' if v else 'FAIL'}")
print("ML-10 self-check:", "PASS" if all(checks.values()) else "FAIL")


ranked_actions: PASS
reason_codes: PASS
archetype_action_mapping: PASS
human_review_required: PASS
automation_disabled: PASS
zero_client_overlap: PASS
queue_exported: PASS
metrics_exported: PASS
ML-10 self-check: PASS
